# 06b. Model Comparison - Automobile Loan Default Prediction

Compares every candidate model using the cross-validation results logged in `experiments/results/`. All numbers come from stratified 5-fold CV on the training split at a 0.5 threshold. The held-out test set is not used in this notebook.


## 1. Setup


In [1]:
import os
import sys

project_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

results_dir = "experiments/results" if os.path.exists("experiments/results") else "../experiments/results"


## 2. Collect Results

Results are read from three files: the shared log, the Logistic Regression file, and the XGBoost baseline-vs-tuned file. Duplicate rows for the same model keep the most recent entry. Each model has a protocol status: `verified` means its CV run is confirmed to match the shared protocol; `unverified` means the source still needs checking.


In [2]:
frames = [
    pd.read_csv(os.path.join(results_dir, "experiments.csv")),
    pd.read_csv(os.path.join(results_dir, "logistic_regression.csv")),
    pd.read_csv(os.path.join(results_dir, "xgboost_baseline_vs_tuned.csv")),
    pd.read_csv(os.path.join(results_dir, "lightgbm_baseline_vs_tuned.csv")),
]
all_results = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['model_name'], keep='last')

PROTOCOL_STATUS = {
    "random_forest": "verified",
    "random_forest_tuned": "verified",
    "gaussian_naive_bayes": "verified",
    "logistic_regression": "verified",
    "XGBoost Baseline": "verified",
    "XGBoost Tuned": "verified",
    "LightGBM Baseline": "verified",
    "LightGBM Tuned": "verified",
    "decision_tree": "verified",
}
all_results["protocol_status"] = all_results["model_name"].map(PROTOCOL_STATUS).fillna("unknown")

print(all_results[['model_name', 'protocol_status', 'roc_auc_mean']].to_string(index=False))


          model_name protocol_status  roc_auc_mean
gaussian_naive_bayes        verified      0.696815
       decision_tree        verified      0.708183
       random_forest        verified      0.774053
 random_forest_tuned        verified      0.769818
 logistic_regression        verified      0.733998
    XGBoost Baseline        verified      0.763093
       XGBoost Tuned        verified      0.770509
   LightGBM Baseline        verified      0.748190
      LightGBM Tuned        verified      0.756597


## 3. Comparison Table

Ranking rule: take the model with the highest remaining mean ROC-AUC. Every model whose mean ROC-AUC is within one of that model's standard deviations joins its tier, because a gap smaller than the fold-to-fold spread is not a real difference. Inside a tier, models are ordered by PR-AUC. Repeat for the remaining models. Recall, precision and F1 are at 0.5.


In [3]:
def mean_pm_std(mean, std):
    return f"{mean:.3f} +/- {std:.3f}"


def rank_with_ties(results):
    remaining = results.sort_values("roc_auc_mean", ascending=False)
    tiers = []
    while not remaining.empty:
        leader = remaining.iloc[0]
        cutoff = leader["roc_auc_mean"] - leader["roc_auc_std"]
        in_tier = remaining[remaining["roc_auc_mean"] >= cutoff]
        tiers.append(in_tier.sort_values("pr_auc_mean", ascending=False).assign(tier=len(tiers) + 1))
        remaining = remaining.drop(in_tier.index)
    return pd.concat(tiers)


ranked = rank_with_ties(all_results)

table = pd.DataFrame({
    "tier": ranked["tier"],
    "model": ranked["model_name"],
    "ROC-AUC": [mean_pm_std(m, s) for m, s in zip(ranked["roc_auc_mean"], ranked["roc_auc_std"])],
    "PR-AUC": [mean_pm_std(m, s) for m, s in zip(ranked["pr_auc_mean"], ranked["pr_auc_std"])],
    "recall": ranked["recall_mean"].round(3),
    "precision": ranked["precision_mean"].round(3),
    "F1": ranked["f1_mean"].round(3),
    "threshold": ranked["threshold"].round(3),
    "protocol_status": ranked["protocol_status"],
}).reset_index(drop=True)
table.insert(0, "rank", range(1, len(table) + 1))
table


,rank,tier,model,ROC-AUC,PR-AUC,recall,precision,F1,threshold,protocol_status
0,1,1,random_forest,0.774 +/- 0.004,0.358 +/- 0.009,0.163,0.632,0.260,0.5,verified
1,2,1,XGBoost Tuned,0.771 +/- 0.004,0.307 +/- 0.009,0.045,0.717,0.085,0.5,verified
2,3,2,random_forest_tuned,0.770 +/- 0.004,0.339 +/- 0.009,0.285,0.381,0.326,0.5,verified
3,4,3,XGBoost Baseline,0.763 +/- 0.004,0.261 +/- 0.010,0.022,0.622,0.042,0.5,verified
4,5,4,LightGBM Tuned,0.757 +/- 0.004,0.248 +/- 0.013,0.560,0.203,0.298,0.5,verified
5,6,5,LightGBM Baseline,0.748 +/- 0.003,0.228 +/- 0.008,0.644,0.168,0.266,0.5,verified
6,7,6,logistic_regression,0.734 +/- 0.005,0.212 +/- 0.005,0.668,0.153,0.249,0.5,verified
7,8,7,decision_tree,0.708 +/- 0.004,0.181 +/- 0.004,0.607,0.152,0.242,0.5,verified
8,9,8,gaussian_naive_bayes,0.697 +/- 0.004,0.176 +/- 0.009,0.479,0.170,0.250,0.5,verified


**Finding:** All nine rows are verified under the same protocol (shared pipeline, `evaluate_model`, stratified 5-fold CV on the training split, threshold 0.5). Under the tiered rule, tier 1 holds Random Forest (untuned) and XGBoost Tuned: their ROC-AUC gap (0.774 vs 0.771) is smaller than Random Forest's standard deviation (0.004), so it is a tie, and PR-AUC puts Random Forest first (0.358 vs 0.307). Random Forest tuned (0.770) is in tier 2 on its own: its gap to the leader (0.0042) is just over one standard deviation (0.0040). The remaining models each form their own tier: XGBoost Baseline (0.763), LightGBM Tuned (0.757), LightGBM Baseline (0.748), Logistic Regression (0.734), Decision Tree (0.708) and Gaussian NB (0.697).

At 0.5, the top models have low recall (0.163 for Random Forest, 0.045 for XGBoost Tuned), while the class-weighted models (LightGBM, Logistic Regression, Decision Tree) have higher recall but much lower precision. Recall at 0.5 depends on the operating threshold, which has not been chosen yet.

**Decision (proposed):** carry the tier-1 models, Random Forest (untuned) and XGBoost Tuned, forward as the leading candidates, with Random Forest first on PR-AUC. The team still needs to confirm this shortlist, agree the operating threshold, and choose the final model before the held-out test set is used.
